In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import os

path = kagglehub.dataset_download("hayaalwizrah1/gold-price-prediction-dataset-20002026")
data = pd.read_csv(os.path.join(path, 'gold_data.csv'))

In [2]:
df = data.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").sort_index()
df.shape

(6468, 24)

In [3]:
df.isnull().sum()

Gold_Open       0
Gold_High       0
Gold_Low        0
Gold_Close      0
Gold_Volume     0
DXY             0
SP500           0
Oil             0
VIX             0
Lag1            0
Lag7            0
MA7             0
MA30            0
EMA20           0
Daily_Return    0
RSI             0
MACD            0
MACD_Signal     0
MACD_Hist       0
BB_Upper        0
BB_Middle       0
BB_Lower        0
ATR             0
Target          0
dtype: int64

In [4]:
# day-of-week / month seasonality, added straight to df so the latest row already has everything it needs later
dow = df.index.dayofweek
df["dow_sin"] = np.sin(2*np.pi*dow/7)
df["dow_cos"] = np.cos(2*np.pi*dow/7)

month = df.index.month
df["month_sin"] = np.sin(2*np.pi*month/12)
df["month_cos"] = np.cos(2*np.pi*month/12)

print(df.shape)

(6468, 28)


In [5]:
df[["Gold_Close","RSI","MACD","DXY","VIX"]].tail()

,Gold_Close,RSI,MACD,DXY,VIX
Date,,,,,
2026-07-24,4067.600098,45.939508,-51.155755,101.470001,18.580000
2026-07-27,4074.500000,46.480858,-46.768670,101.510002,18.670000
2026-07-28,4036.300049,43.862116,-45.845807,101.379997,18.209999
2026-07-29,4034.699951,43.750924,-44.727951,100.800003,20.660000
2026-07-30,4100.100098,49.397363,-38.125311,100.010002,17.090000


In [6]:
df.columns

Index(['Gold_Open', 'Gold_High', 'Gold_Low', 'Gold_Close', 'Gold_Volume',
       'DXY', 'SP500', 'Oil', 'VIX', 'Lag1', 'Lag7', 'MA7', 'MA30', 'EMA20',
       'Daily_Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper',
       'BB_Middle', 'BB_Lower', 'ATR', 'Target', 'dow_sin', 'dow_cos',
       'month_sin', 'month_cos'],
      dtype='str')

In [7]:
HORIZON = 21  # ~1 trading month ahead (21 trading days)
df['Target_Direct'] = df["Gold_Close"].shift(-HORIZON)
df = df.dropna(subset=['Target', 'Target_Direct'])

X = df.drop(columns=['Target', 'Target_Direct'])

y_Recursive = df['Target']      # tomorrow's Close
y_Direct = df['Target_Direct']  # Close after 21 days

In [8]:
n = len(X)

train_end = int(n * 0.70)
val_end = int(n * 0.80)   # 70% train + 10% validation

# X split
X_train = X.iloc[:train_end]
X_val   = X.iloc[train_end:val_end]
X_test  = X.iloc[val_end:]

# Recursive target split
y_r_train = y_Recursive.iloc[:train_end]
y_r_val   = y_Recursive.iloc[train_end:val_end]
y_r_test  = y_Recursive.iloc[val_end:]

# Direct target split
y_d_train = y_Direct.iloc[:train_end]
y_d_val   = y_Direct.iloc[train_end:val_end]
y_d_test  = y_Direct.iloc[val_end:]

# Test dates
dates_test = df.index[val_end:]

print("x Train:", X_train.shape)
print("x Validation:", X_val.shape)
print("x Test:", X_test.shape)

print("\nRecursive:")
print("y Train:", y_r_train.shape)
print("y Validation:", y_r_val.shape)
print("y Test:", y_r_test.shape)

print("\nDirect:")
print("y Train:", y_d_train.shape)
print("y Validation:", y_d_val.shape)
print("y Test:", y_d_test.shape)

x Train: (4512, 27)
x Validation: (645, 27)
x Test: (1290, 27)

Recursive:
y Train: (4512,)
y Validation: (645,)
y Test: (1290,)

Direct:
y Train: (4512,)
y Validation: (645,)
y Test: (1290,)


In [9]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_val_s   = scaler_X.transform(X_val)
X_test_s  = scaler_X.transform(X_test)

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Recursive Baselines
naive_recursive_pred = X_test["Gold_Close"].values
linreg_recursive = LinearRegression()
linreg_recursive.fit(X_train_s,y_r_train)

linreg_recursive_pred_s = linreg_recursive.predict(X_test_s)

def evaluate_baseline(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(
        f"{label:35s}"
        f"MAE={mae:8.3f}  "
        f"RMSE={rmse:8.3f}  "
        f"R²={r2:7.4f}"
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

recursive_baseline_results = {}

recursive_baseline_results["naive"] = evaluate_baseline(y_r_test.values, naive_recursive_pred, "Recursive Naive Baseline")
recursive_baseline_results["linreg"] = evaluate_baseline(y_r_test.values, linreg_recursive_pred_s, "Recursive Linear Regression")

# Direct Baselines --------------------------------------------

naive_direct_pred = X_test["Gold_Close"].values
linreg_direct = LinearRegression()
linreg_direct.fit(X_train_s,y_d_train)

linreg_direct_pred_s = linreg_direct.predict(X_test_s)

direct_baseline_results = {}

direct_baseline_results["naive"] = evaluate_baseline(y_d_test.values, naive_direct_pred, "Direct Naive Baseline")
direct_baseline_results["linreg"] = evaluate_baseline(y_d_test.values, linreg_direct_pred_s, "Direct Linear Regression")

Recursive Naive Baseline           MAE=  23.170  RMSE=  41.214  R²= 0.9981
Recursive Linear Regression        MAE=  23.818  RMSE=  41.574  R²= 0.9981
Direct Naive Baseline              MAE= 107.302  RMSE= 162.796  R²= 0.9720
Direct Linear Regression           MAE= 131.907  RMSE= 193.157  R²= 0.9605


In [11]:
from GoldPricePredictor import GoldPricePredictor

predictor = GoldPricePredictor()

In [ ]:
predictor.train_recursive(
    X_train=X_train_s,
    y_train=y_r_train,
    X_val=X_val_s,
    y_val=y_r_val,
    epochs=300,
    batch_size=32,
    tune_epochs=80,
    tune_patience=10,
    patience=20
)

--------- Training Recursive Model --------------

Config 0: {'units1': 64, 'units2': 32, 'units3': 16, 'dropout1': 0.2, 'dropout2': 0.1, 'l2reg': 0.0001, 'lr': 0.001} -> val_mae=16.9829
Config 1: {'units1': 128, 'units2': 64, 'units3': 32, 'dropout1': 0.3, 'dropout2': 0.2, 'l2reg': 0.001, 'lr': 0.001} -> val_mae=21.6329


In [ ]:
predictor.train_direct(
    X_train=X_train_s,
    y_train=y_d_train,
    X_val=X_val_s,
    y_val=y_d_val,
    epochs=300,
    batch_size=32,
    tune_epochs=80,
    tune_patience=10,
    patience=20
)

In [ ]:
feature_names = X.columns.tolist()

predictor.recursive_model.features = feature_names
predictor.direct_model.features = feature_names

In [ ]:
recursive_results = predictor.evaluate_recursive(
    X_test=X_test_s,
    y_test=y_r_test,
)

direct_results = predictor.evaluate_direct(
    X_test=X_test_s,
    y_test=y_d_test,
)

In [ ]:
recursive_pred_s = predictor.recursive_model.model.predict(X_test_s, verbose=0).ravel()

comparison = pd.DataFrame({
    "Model": ["Naive","Linear Regression","ANN"],
    "MAE": [
        recursive_baseline_results["naive"]["MAE"],
        recursive_baseline_results["linreg"]["MAE"],
        recursive_results["MAE"]
    ],
    "RMSE": [
        recursive_baseline_results["naive"]["RMSE"],
        recursive_baseline_results["linreg"]["RMSE"],
        recursive_results["RMSE"]
    ],
    "R2": [
        recursive_baseline_results["naive"]["R2"],
        recursive_baseline_results["linreg"]["R2"],
        recursive_results["R2"]
    ]
})
print(comparison)

plt.figure(figsize=(12,5))
plt.plot(dates_test, y_r_test.values, label="Actual",alpha=0.6)
plt.plot(dates_test, recursive_pred_s, label="ANN",alpha=0.6)
plt.plot(dates_test, naive_recursive_pred, label="Naive",alpha=0.6)
plt.plot(dates_test, linreg_recursive_pred_s, label="Linear Regression", alpha=0.8)
plt.legend()
plt.title("Recursive Forecast Comparison")
plt.show()

In [ ]:
direct_pred_s = predictor.direct_model.model.predict(X_test_s, verbose=0).ravel()

comparison = pd.DataFrame({
    "Model": ["Naive","Linear Regression","ANN"],
    "MAE": [
        direct_baseline_results["naive"]["MAE"],
        direct_baseline_results["linreg"]["MAE"],
        direct_results["MAE"]
    ],
    "RMSE": [
        direct_baseline_results["naive"]["RMSE"],
        direct_baseline_results["linreg"]["RMSE"],
        direct_results["RMSE"]
    ],
    "R2": [
        direct_baseline_results["naive"]["R2"],
        direct_baseline_results["linreg"]["R2"],
        direct_results["R2"]
    ]
})
print(comparison)

plt.figure(figsize=(12,5))
plt.plot(dates_test, y_d_test.values, label="Actual")
plt.plot(dates_test,direct_pred_s, label="ANN")
plt.plot(dates_test, naive_direct_pred, label="Naive", alpha=0.6)
plt.plot( dates_test, linreg_recursive_pred_s, label="Linear Regression", alpha=0.8)
plt.legend()
plt.title("Direct Forecast Comparison (21 Days Ahead)")
plt.xlabel("Date")
plt.ylabel("Gold Price")
plt.grid(True)
plt.show()

In [ ]:
def predict_gold(date):

    row = df.loc[pd.to_datetime(date)]

    recursive_result = predictor.predict_recursive(
        latest_data=row,
        historical_data=df.loc[:date],
        scaler_X=scaler_X,
        days=21
    )

    direct_result = predictor.direct_model.predict_after_21_days(
        latest_data=row,
        scaler_X=scaler_X,
        current_price=row["Gold_Close"]
    )

    return {
        "recursive": recursive_result,
        "direct": direct_result
    }

In [ ]:
result = predict_gold("2025-01-15")

In [ ]:
forecast_df = result["recursive"]

print(forecast_df.head())
print(forecast_df.tail())